# Module 3 — Complete Tampering Detection Pipeline

This notebook runs the complete end-to-end Module 3 (EfficientNet-B0 Tampering Detection) pipeline:
1. **Step 1**: Build / Re-generate Synthetic Tampering Dataset
2. **Step 2**: Load Grouped Dataset Splits (Train / Val / Test)
3. **Step 3**: Train EfficientNet-B0 Patch Classifier
4. **Step 4**: Evaluate Model on Held-Out Test Split (ROC-AUC, Precision, Recall, F1, Confusion Matrix)
5. **Step 5**: Test Inference Prediction on a Sample Document

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

Project Root: C:\Users\priya\Desktop\prototype


## Step 1: Re-build Synthetic Tampering Dataset

In [2]:
from app.tampering.dataset_builder import build_dataset, validate_dataset

# Source directory for passport images
source_dir = project_root / "data" / "samples"
if not source_dir.exists() or not list(source_dir.glob("*")):
    source_dir = project_root / "passport_images"

dataset_root = project_root / "data" / "tampering_dataset"

print(f"Building dataset from '{source_dir}' -> '{dataset_root}'...")
build_res = build_dataset(
    source_dir=source_dir,
    dataset_root=dataset_root,
    seed=42
)

print("Build Success:", build_res["success"])
print("Full Manifest:", build_res.get("full_manifest_path"))

val_res = validate_dataset(dataset_root)
print("Validation Summary:", val_res)

Building dataset from 'C:\Users\priya\Desktop\prototype\passport_images' -> 'C:\Users\priya\Desktop\prototype\data\tampering_dataset'...
Build Success: True
Full Manifest: C:\Users\priya\Desktop\prototype\data\tampering_dataset\manifests\dataset_manifest.csv
Validation Summary: {'valid': True, 'genuine_count': 200, 'tampered_count': 1400, 'missing_files': [], 'invalid_labels': [], 'duplicates': [], 'errors': [], 'total': 1600}


## Step 2: Load Grouped Dataset Splits (Train / Val / Test)

In [3]:
from app.tampering.dataset_loader import load_dataset, check_dataset_readiness

readiness = check_dataset_readiness(dataset_root)
print("Dataset Readiness:", readiness)

split_res = load_dataset(dataset_root, train_ratio=0.70, val_ratio=0.15, seed=42)
print("Split Success:", split_res["success"])
print("Class Counts :", split_res.get("class_counts"))
print("Split Counts :", split_res.get("split_counts"))

Dataset Readiness: {'ready': True, 'genuine_count': 200, 'tampered_count': 1400, 'has_manifest': True, 'issues': [], 'dataset_root': 'C:\\Users\\priya\\Desktop\\prototype\\data\\tampering_dataset'}
Split Success: True
Class Counts : {'GENUINE': 200, 'TAMPERED': 1400}
Split Counts : {'train': 1120, 'val': 240, 'test': 240}


## Step 3: Train EfficientNet-B0 Patch Classifier

In [4]:
from app.tampering.train import train

ckpt_dir = project_root / "data" / "model_checkpoints" / "efficientnet_b0"

print("Starting EfficientNet-B0 training...")
train_res = train(
    train_samples=split_res["train"],
    val_samples=split_res["val"],
    checkpoint_dir=ckpt_dir,
    pretrained=True,
    freeze_backbone=True,
    unfreeze_after=5,
    lr=1e-4,
    batch_size=16,
    epochs=15,
    patience=5,
    seed=42,
)

print("\n--- Training Complete ---")
print("Best Val Accuracy:", train_res.get("best_val_acc"))
print("Best Epoch       :", train_res.get("best_epoch"))
print("Checkpoint Path  :", train_res.get("checkpoint_path"))

Starting EfficientNet-B0 training...
Epoch   1/15  train_loss=0.6662  train_acc=0.6027  val_loss=0.6507  val_acc=0.6331  (210.4s)
  ✓ New best val_acc=0.6331 — checkpoint saved.
Epoch   2/15  train_loss=0.6394  train_acc=0.6661  val_loss=0.6355  val_acc=0.6560  (192.8s)
  ✓ New best val_acc=0.6560 — checkpoint saved.
Epoch   3/15  train_loss=0.6195  train_acc=0.6937  val_loss=0.6150  val_acc=0.6831  (193.0s)
  ✓ New best val_acc=0.6831 — checkpoint saved.
Epoch   4/15  train_loss=0.6079  train_acc=0.6946  val_loss=0.6052  val_acc=0.6971  (191.7s)
  ✓ New best val_acc=0.6971 — checkpoint saved.
Epoch   5/15  train_loss=0.5780  train_acc=0.7402  val_loss=0.5785  val_acc=0.7302  (278.1s)
  ✓ New best val_acc=0.7302 — checkpoint saved.
Epoch   6/15  train_loss=0.5528  train_acc=0.7527  val_loss=0.5519  val_acc=0.7606  (274.3s)
  ✓ New best val_acc=0.7606 — checkpoint saved.
Epoch   7/15  train_loss=0.5130  train_acc=0.7937  val_loss=0.5312  val_acc=0.7819  (269.8s)
  ✓ New best val_acc=0.7

## Step 4: Evaluate Model on Held-Out Test Split (ROC-AUC & Metrics)

In [5]:
import importlib
import app.tampering.evaluate
importlib.reload(app.tampering.evaluate)
from app.tampering.evaluate import evaluate

ckpt_path = ckpt_dir / "best_checkpoint.pt"
output_dir = project_root / "data" / "evaluation_results"

eval_res = evaluate(
    test_samples=split_res["test"],
    checkpoint_path=ckpt_path,
    output_dir=output_dir,
)

print("=== HELD-OUT TEST EVALUATION RESULT ===")
print("Accuracy :", eval_res.get("accuracy"))
print("Precision:", eval_res.get("precision"))
print("Recall   :", eval_res.get("recall"))
print("F1-Score :", eval_res.get("f1"))
print("ROC-AUC  :", eval_res.get("roc_auc"))
print("\nConfusion Matrix:\n", eval_res.get("confusion_matrix"))
print("\nClassification Report:\n", eval_res.get("classification_report"))

Results saved → C:\Users\priya\Desktop\prototype\data\evaluation_results\test_evaluation.json, C:\Users\priya\Desktop\prototype\data\evaluation_results\per_sample_predictions.csv
=== HELD-OUT TEST EVALUATION RESULT ===
Accuracy : 0.82917
Precision: 0.87308
Recall   : 0.82917
F1-Score : 0.84566
ROC-AUC  : 0.87571

Confusion Matrix:
 [[19, 11], [30, 180]]

Classification Report:
               precision    recall  f1-score   support

     GENUINE       0.39      0.63      0.48        30
    TAMPERED       0.94      0.86      0.90       210

    accuracy                           0.83       240
   macro avg       0.67      0.75      0.69       240
weighted avg       0.87      0.83      0.85       240



## Step 5: Run Sample Inference Prediction

In [6]:
from app.tampering.inference import predict

if split_res["test"]:
    sample_img = split_res["test"][0]["image_path"]
    print("Predicting for sample:", sample_img)
    pred_res = predict(image_path=sample_img, checkpoint_path=ckpt_path)
    
    print("\n=== INFERENCE PREDICTION ===")
    print("Decision Band     :", pred_res.get("band"))
    print("Tampered Score    :", pred_res.get("score_tampered"))
    print("Genuine Score     :", pred_res.get("score_genuine"))
    print("Total Patches     :", pred_res.get("n_patches"))
    print("Suspicious Regions:", pred_res.get("suspicious_regions"))

Predicting for sample: C:\Users\priya\Desktop\prototype\data\tampering_dataset\images\genuine\passport_115\passport_115_genuine_original.jpg

=== INFERENCE PREDICTION ===
Decision Band     : UNCERTAIN
Tampered Score    : 0.57959
Genuine Score     : 0.42041
Total Patches     : 60
Suspicious Regions: [{'box': [544, 784, 768, 1008], 'score': 0.6514}, {'box': [544, 976, 768, 1200], 'score': 0.5471}, {'box': [544, 896, 768, 1120], 'score': 0.5402}]
